# 02 Classifier V2: compare the three loss families
* **wbce**  – normal class-weighted BCE
* **lb**    – WBCE + lesion-vs-background separation (consistency, background suppression, margin on the full / lesion-removed views)
* **pheno** – two-branch phenotype objective (WBCE + attribute + pair + orthogonality)
* **full**  – all terms (and a lighter-weighted variant)
Cosine learning-rate schedule and flip test-time augmentation are used for every run. The best run on validation AUC becomes `clf_v2`.

In [2]:
from google.colab import drive; drive.mount("/content/drive")
import glob, os, sys, json
hits = glob.glob("/content/drive/MyDrive/mammo_semcom/**/src/config.py", recursive=True)
CODE_DIR = os.path.dirname(os.path.dirname(hits[0])); WORK_DIR = "/content/drive/MyDrive/mammo_semcom_work"
sys.path.insert(0, CODE_DIR); print("code:", CODE_DIR)
import shutil, numpy as np, pandas as pd
from src.config import Cfg
from src import data
from src.metrics import auc_safe
from src.train_v2 import descriptors, train_classifier_v2, ClfDS, predict_lesion

ZIP_PATH = "/content/drive/MyDrive/mammo_semcom_data/CBIS-DDSM.zip"
RUNS = {"wbce": dict(mode="wbce"), "lb": dict(mode="lb"), "pheno": dict(mode="pheno"), "full": dict(mode="full"),
        "full_light": dict(mode="full", lam=dict(attr=.2, pair=.05, orth=.05, F=.3, C=.3, B=.3, M=.3))}       # delete entries to save time
cfg = Cfg(work_dir=WORK_DIR, clf_epochs=20).ensure()
images, masks, images_df, abn_df = data.load_cache(cfg)
heat = np.load(cfg.path("cache", "heatmaps_v2.npy"))
A, K = descriptors(ZIP_PATH, abn_df, images_df)
te = np.where(abn_df.split.values == "test")[0]
test_ds = ClfDS(images, heat, abn_df.iloc[te], A[te], cfg, train=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
code: /content/drive/MyDrive/mammo_semcom
descriptor observed per column: {'mass_shape': 1692, 'mass_margins': 1636, 'calc_type': 1847, 'calc_distribution': 1432} of 3567


In [3]:
res = {}
for name, kw in RUNS.items():
    m, thr, best_val = train_classifier_v2(cfg, images, heat, abn_df, A, K, name=name, **kw)
    p = predict_lesion(m, test_ds)
    res[name] = dict(val_AUC=best_val, test_AUC=auc_safe(test_ds.y, p), test_ACC=float(((p >= thr) == test_ds.y).mean()), thr=thr)
    print(name, {a: round(b, 3) for a, b in res[name].items()})
table = pd.DataFrame(res).T.astype(float); display(table.round(3))
best = table.val_AUC.idxmax(); print("best on validation:", best)
for ext in ("pt", "json"):
    shutil.copy(cfg.path("models", f"clf_{best}.{ext}"), cfg.path("models", f"clf_v2.{ext}"))
table.to_csv(cfg.path("results", "classifier_loss_comparison.csv"))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 166MB/s]


[wbce] epoch  1/20 | loss 0.776 | val AUC 0.7268
[wbce] epoch  2/20 | loss 0.708 | val AUC 0.7471
[wbce] epoch  3/20 | loss 0.753 | val AUC 0.7354
[wbce] epoch  4/20 | loss 0.726 | val AUC 0.7567
[wbce] epoch  5/20 | loss 0.718 | val AUC 0.7834
[wbce] epoch  6/20 | loss 0.623 | val AUC 0.7558
[wbce] epoch  7/20 | loss 0.679 | val AUC 0.7659
[wbce] epoch  8/20 | loss 0.717 | val AUC 0.7623
[wbce] epoch  9/20 | loss 0.573 | val AUC 0.7588
[wbce] epoch 10/20 | loss 0.701 | val AUC 0.7395
[wbce] epoch 11/20 | loss 0.573 | val AUC 0.7588
[wbce] epoch 12/20 | loss 0.583 | val AUC 0.7594
[wbce] epoch 13/20 | loss 0.699 | val AUC 0.7644
[wbce] epoch 14/20 | loss 0.593 | val AUC 0.7741
[wbce] epoch 15/20 | loss 0.674 | val AUC 0.7635
[wbce] epoch 16/20 | loss 0.467 | val AUC 0.7669
[wbce] epoch 17/20 | loss 0.548 | val AUC 0.7700
[wbce] epoch 18/20 | loss 0.684 | val AUC 0.7701
[wbce] epoch 19/20 | loss 0.504 | val AUC 0.7692
[wbce] epoch 20/20 | loss 0.675 | val AUC 0.7696
wbce {'val_AUC': 0.7

,val_AUC,test_AUC,test_ACC,thr
wbce,0.783,0.746,0.666,0.405
lb,0.750,0.699,0.680,0.750
pheno,0.798,0.756,0.681,0.347
full,0.761,0.723,0.689,0.677
full_light,0.748,0.712,0.674,0.726


best on validation: pheno
